# Score-weight sensitivity analysis — 600 cases

This notebook evaluates six persona-quality weights across eight multipliers on the full 600-case benchmark. It excludes pilot, threshold-sensitivity, debugging, and alternative run modes. Set the environment variables described below before running. API calls incur costs. The 600 test opinions are embedded in this notebook.

In [ ]:
!pip install -q pandas numpy tqdm torch transformers sentencepiece accelerate openai

In [ ]:
import os
import re
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from openai import OpenAI

In [ ]:
from openai import AzureOpenAI

# Configure credentials in your environment
required_env = ["AZURE_OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT"]
missing = [name for name in required_env if not os.getenv(name)]
if missing:
    raise EnvironmentError(f"Missing required environment variables: {', '.join(missing)}")
client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")


In [ ]:
# Run from the repository root, or set PROJECT_DATA_DIR to your data directory.
BASE_DIR = Path("/content")
OUT_DIR = BASE_DIR / "results" / "sensitivity"
REPOSITORY_DIR = BASE_DIR / "data" / "final_topic_metas"
BASELINE_META_DIR = REPOSITORY_DIR
BASELINE_META_FILES = {"blm": "blm_meta.parquet", "guns": "guns_meta.parquet", "samesex": "samesex_meta.parquet"}
EVAL_OUT_DIR = OUT_DIR
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
TOPICS = ["blm", "guns", "samesex"]
RESULT_SUFFIX = "expanded_all_topics_600_cases"
FULL_RESULTS_PATH = EVAL_OUT_DIR / f"sensitivity_full_results_{RESULT_SUFFIX}.csv"
OFFLINE_ABLATION_VARIANTS = ["full_method"]
METHOD_NLI_MODEL_NAME = "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"
EVAL_NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
GPT_MODEL = AZURE_OPENAI_DEPLOYMENT
POOL_SIZE = 60
K2_SHORT = 8
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Evaluation cases: 600; topics:", TOPICS)
print("Device:", device, "Repository:", REPOSITORY_DIR)


In [ ]:
def normalize_topic(topic):
    topic = str(topic).lower().strip()

    if topic in ["blm", "black lives matter"]:
        return "blm"

    if topic in ["guns", "gun", "gun control", "gun_control"]:
        return "guns"

    if topic in ["samesex", "same-sex", "same sex", "same-sex marriage", "same sex marriage"]:
        return "samesex"

    return topic


def topic_name(topic):
    topic = normalize_topic(topic)

    if topic == "blm":
        return "Black Lives Matter"

    if topic == "guns":
        return "gun control"

    if topic == "samesex":
        return "same-sex marriage legalization"

    return topic


def normalize_stance(x):
    x = str(x).lower().strip()

    if x in ["pro", "for", "support", "supports", "supporting"]:
        return "pro"

    if x in ["anti", "against", "oppose", "opposes", "opposing"]:
        return "anti"

    return x


def opposite_stance(stance):
    stance = normalize_stance(stance)

    if stance == "pro":
        return "anti"

    if stance == "anti":
        return "pro"

    return "unknown"


def detect_text_col(df):
    for col in ["_text", "comment_text", "text", "message"]:
        if col in df.columns:
            return col
    raise ValueError("No text column found.")


def clean_json_text(text):
    text = str(text).strip()
    text = text.replace("```json", "").replace("```", "").strip()
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)
    return text


def parse_json(text):
    return json.loads(clean_json_text(text))


def call_gpt(prompt, system="You are a helpful assistant.", max_tokens=15000, retries=3):
    last_error = None

    model_name = globals().get("AZURE_OPENAI_DEPLOYMENT", GPT_MODEL)

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": prompt},
                ],
                max_completion_tokens=max_tokens,
            )

            content = response.choices[0].message.content

            if content is None or not str(content).strip():
                raise ValueError(f"Empty GPT response. Full response: {response}")

            return str(content).strip()

        except Exception as e:
            last_error = e
            print(f"GPT attempt {attempt + 1}/{retries} failed:", repr(e))
            time.sleep(2)

    raise RuntimeError(f"GPT failed after {retries} attempts: {repr(last_error)}")


def call_gpt_json(prompt, system="Return valid JSON only.", max_tokens=15000, retries=3):
    content = call_gpt(prompt, system=system, max_tokens=max_tokens, retries=retries)
    return parse_json(content)

In [ ]:
method_tokenizer = AutoTokenizer.from_pretrained(METHOD_NLI_MODEL_NAME)
method_nli_model = AutoModelForSequenceClassification.from_pretrained(
    METHOD_NLI_MODEL_NAME
).to(device)
method_nli_model.eval()

eval_tokenizer = AutoTokenizer.from_pretrained(EVAL_NLI_MODEL_NAME)
eval_nli_model = AutoModelForSequenceClassification.from_pretrained(
    EVAL_NLI_MODEL_NAME
).to(device)
eval_nli_model.eval()


def find_contradiction_index(model):
    for idx, label in model.config.id2label.items():
        if "contradiction" in str(label).lower():
            return int(idx)
    raise ValueError(model.config.id2label)


METHOD_CONTRA_IDX = find_contradiction_index(method_nli_model)
EVAL_CONTRA_IDX = find_contradiction_index(eval_nli_model)

print("Method NLI:", METHOD_NLI_MODEL_NAME, method_nli_model.config.id2label)
print("Eval NLI:", EVAL_NLI_MODEL_NAME, eval_nli_model.config.id2label)

In [ ]:
@torch.inference_mode()
def contradiction_scores_batch(
    premise,
    hypotheses,
    tokenizer,
    model,
    contradiction_idx,
    batch_size=16,
    max_length=512,
):
    scores = []

    hypotheses = [str(h) for h in hypotheses]

    for i in range(0, len(hypotheses), batch_size):
        batch_h = hypotheses[i:i + batch_size]
        batch_p = [str(premise)] * len(batch_h)

        encoded = tokenizer(
            batch_p,
            batch_h,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length,
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        logits = model(**encoded).logits
        probs = torch.softmax(logits, dim=-1)

        scores.extend(
            probs[:, contradiction_idx]
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

    return np.array(scores, dtype=np.float32)


def method_nli_scores(user_opinion, candidate_texts):
    return contradiction_scores_batch(
        premise=user_opinion,
        hypotheses=candidate_texts,
        tokenizer=method_tokenizer,
        model=method_nli_model,
        contradiction_idx=METHOD_CONTRA_IDX,
    )


def eval_nli_score(user_opinion, output_text):
    if not str(output_text).strip():
        return np.nan

    return float(
        contradiction_scores_batch(
            premise=user_opinion,
            hypotheses=[output_text],
            tokenizer=eval_tokenizer,
            model=eval_nli_model,
            contradiction_idx=EVAL_CONTRA_IDX,
        )[0]
    )

In [ ]:
# ============================================================
# 1. Expanded balanced test set: 180 cases
# 3 topics × 2 stances × 30 examples
# ============================================================

def make_cases(topic, stance, opinions):
    return [
        {
            "topic": topic,
            "user_opinion": opinion,
            "expected_user_stance": stance,
        }
        for opinion in opinions
    ]



BLM_PRO = ['Black Lives Matter is important because police violence and racial discrimination are real problems that need public '
 'attention.',
 'I support BLM because Black communities still face systemic racism, and ignoring that only protects the status quo.',
 'BLM is necessary because racial injustice did not disappear, and people need to protest when institutions fail them.',
 'The movement matters because police accountability and racial equality should not be controversial ideas.',
 'I think BLM helped expose problems that many people preferred to ignore, especially around policing and '
 'discrimination.',
 'BLM gives people a way to demand justice when official institutions do not take police violence seriously.',
 'I support BLM because equal treatment under the law is still not experienced equally by everyone.',
 'The protests were justified because people were tired of seeing repeated cases of police brutality without '
 'accountability.',
 'BLM is not about hating anyone; it is about saying that Black lives should be protected and valued equally.',
 'I think the movement is important because racial inequality still affects policing, housing, education, and '
 'opportunity.',
 'BLM brought attention to injustice that many communities have been talking about for decades.',
 'Supporting BLM means supporting accountability, fairness, and equal protection for Black citizens.',
 'I believe BLM is needed because silence about racism allows unfair systems to continue.',
 'The movement helped make police violence and systemic discrimination part of the national conversation.',
 'I support BLM because people should not have to accept discrimination as normal.',
 'BLM is a legitimate response to real cases where Black people were treated unfairly or violently.',
 'I think criticizing BLM ignores the deeper problem of racial injustice that the movement is trying to address.',
 'The slogan Black Lives Matter is not saying other lives do not matter; it is highlighting a group that has been '
 'harmed.',
 'BLM is important because public pressure is often necessary before institutions make meaningful reforms.',
 'I support the movement because everyone should care when a community repeatedly faces unequal treatment.',
 'BLM represents a demand for dignity, fairness, and accountability in policing.',
 'I think the movement gave a voice to people who felt ignored by courts, police departments, and politicians.',
 'BLM is necessary because racism can exist in systems even when individuals deny being racist.',
 'The movement matters because police accountability should be a basic democratic principle.',
 'I support BLM because racial justice is not a radical demand.',
 'BLM helped show that many people experience law enforcement differently depending on race.',
 'I think the movement is justified because injustice should be confronted publicly, not hidden.',
 'BLM is important because communities need protection from both crime and abusive policing.',
 'I support BLM because equality requires recognizing when one group faces specific harms.',
 'The movement is necessary because without protest, many cases of injustice would be forgotten.',
 'I support BLM because racial disparities in access to quality legal representation can affect whether people receive '
 'equal treatment in the justice system.',
 'The movement encouraged conversations about criminal justice reform that had been ignored for many years.',
 'BLM is important because unequal opportunities in education and housing are also part of racial inequality.',
 'I believe racial inequality affects economic opportunities, so BLM is about more than policing.',
 'The movement helped increase public awareness about discrimination in employment and hiring.',
 'Supporting BLM means recognizing that equal legal rights do not always translate into equal opportunities.',
 'I think BLM encouraged governments and institutions to reconsider policies that unintentionally disadvantage '
 'minority communities.',
 'The protests showed that many citizens are willing to participate peacefully to demand institutional reform.',
 'BLM helped encourage discussions about improving relationships between police departments and local communities.',
 'I support BLM because communities deserve transparency when investigating police misconduct.',
 'The movement encouraged many organizations to examine their own diversity and inclusion policies.',
 'BLM highlighted that historical inequalities can continue affecting people even after laws change.',
 'I think addressing racial inequality requires both policing reforms and broader social investment.',
 'Supporting BLM means supporting equal opportunities in education, employment, and justice.',
 'The movement helped many people learn about historical patterns of discrimination that they had never studied.',
 'I support BLM because independent investigations of police misconduct are necessary when departments cannot credibly '
 'investigate themselves.',
 'BLM has helped make body-camera policies and transparency in policing part of a broader public discussion.',
 'I think qualified immunity deserves reform because accountability is difficult when public officials are rarely held '
 'personally responsible for misconduct.',
 'The movement is important because communities should have a meaningful role in deciding how they are policed.',
 'I support BLM because police departments should publish clearer data about stops, searches, use of force, and '
 'disciplinary actions.',
 'BLM is relevant beyond police shootings because racial disparities can also appear in sentencing, bail, and '
 'incarceration.',
 'I support the movement because criminal justice reform should examine whether similar offenses receive similar '
 'treatment across racial groups.',
 'BLM helped raise awareness that cash bail can punish poor defendants before they have even been convicted.',
 'I think the movement has contributed to useful discussions about rehabilitation instead of relying only on '
 'punishment.',
 'Supporting BLM includes questioning policies that create lasting consequences for minor criminal convictions.',
 'BLM matters because historical housing discrimination still affects where families can live and what opportunities '
 'they have.',
 'I support BLM because racial gaps in home ownership did not appear by accident and deserve serious policy attention.',
 'The movement highlights how neighborhood segregation can affect access to schools, transportation, and employment.',
 'I think fair-housing enforcement is part of racial justice because discrimination in renting and lending still '
 'occurs.',
 'BLM encourages people to consider how older policies such as redlining can have effects that continue across '
 'generations.',
 'I support BLM because students should have equal access to well-funded schools regardless of the racial composition '
 'of their neighborhood.',
 'The movement is important because racial disparities in school discipline deserve examination rather than being '
 'automatically accepted.',
 'BLM has encouraged schools to discuss whether their curricula adequately represent Black history and experiences.',
 'I think racial equality requires addressing differences in educational resources, not only legal discrimination.',
 'Supporting BLM means taking achievement gaps seriously while examining the social conditions that help create them.',
 'BLM is relevant to employment because racial discrimination in hiring can limit opportunities even when applicants '
 'have similar qualifications.',
 'I support efforts inspired by BLM to make hiring and promotion practices more transparent.',
 'The racial wealth gap deserves attention because wealth affects education, housing, health, and the ability to '
 'recover from emergencies.',
 'I think racial justice should include improving access to capital and business opportunities in historically '
 'disadvantaged communities.',
 'BLM helped broaden the discussion from individual prejudice to economic structures that can reproduce inequality.',
 'I support BLM because racial disparities in maternal mortality and other health outcomes deserve urgent attention.',
 'The movement is relevant to healthcare when Black patients report that their pain or symptoms are not always taken '
 'as seriously.',
 'Racial justice includes making sure that quality healthcare is available regardless of neighborhood or income.',
 'BLM encouraged more discussion of how environmental hazards can disproportionately affect minority communities.',
 'I think public health institutions should investigate racial disparities instead of assuming that different outcomes '
 'have purely individual causes.',
 'I support BLM because peaceful protest is a legitimate democratic response when people believe institutions are not '
 'listening.',
 'Large BLM demonstrations showed that racial justice concerns were shared by people from many different backgrounds.',
 'I think disruptive protest can sometimes be justified when ordinary political channels have repeatedly failed to '
 'produce reform.',
 'The movement demonstrates why freedom of assembly is important even when protests make other people uncomfortable.',
 'BLM has encouraged younger citizens to become more involved in voting, local politics, and community organizing.',
 'I support BLM because discrimination can result from institutional rules even when no individual explicitly intends '
 'to discriminate.',
 'The concept of systemic racism is useful because it asks us to examine outcomes and institutions rather than only '
 'personal attitudes.',
 'BLM encourages organizations to measure whether their policies produce unequal outcomes and to investigate why.',
 'I think acknowledging structural inequality does not mean accusing every individual person of being racist.',
 'The movement provides a framework for discussing how historical disadvantages can persist after discriminatory laws '
 'are formally removed.',
 'BLM has made people think more carefully about how Black Americans are represented in news, entertainment, and '
 'public institutions.',
 'I support the movement because stereotypes can influence how people are treated even when discrimination is subtle.',
 'The movement has encouraged museums, schools, and public organizations to examine whose experiences are included in '
 'national history.',
 'I think representation matters because people are more likely to trust institutions when they can see themselves '
 'included in them.',
 'BLM has helped create space for Black communities to describe their own experiences instead of having others define '
 'those experiences for them.',
 'I support BLM even though I do not agree with every proposal associated with the movement, because its central '
 'concerns about racial inequality are legitimate.',
 'Supporting BLM does not require supporting the abolition of police; meaningful police reform can still be consistent '
 "with the movement's goals.",
 "I think it is possible to support both effective law enforcement and BLM's demand for greater accountability.",
 'The movement should be judged by the problems it highlights, not only by the most radical statements made by '
 'individual activists.',
 'I support BLM because reform movements can be valuable even when their supporters disagree about specific solutions.',
 "BLM matters because equal treatment should be evaluated through people's actual experiences, not only through what "
 'laws say on paper.',
 'I support the movement because democracy works better when marginalized communities believe institutions take their '
 'concerns seriously.',
 'Racial justice benefits society as a whole because institutions that are fair and accountable are better for '
 'everyone.',
 'I think addressing racial inequality can strengthen social trust rather than weaken it.',
 'BLM is important because equal opportunity requires identifying barriers that affect particular communities rather '
 'than pretending everyone starts from the same position.']

BLM_ANTI = ['BLM is nonsense and only creates division between people who should be treated equally.',
 'I do not support BLM because I think it exaggerates racism and turns every problem into a racial issue.',
 'The movement caused more anger than solutions, and I do not think it actually helped society.',
 'I think BLM unfairly attacks police and ignores personal responsibility in communities.',
 'BLM seems more like political activism than a real solution to crime or social problems.',
 'I oppose BLM because it makes people focus on race instead of treating everyone as individuals.',
 'I think the movement blames police too quickly and ignores how dangerous their job can be.',
 'BLM has become too political and divisive to be taken seriously as a justice movement.',
 'I do not support BLM because I believe all lives matter and no group should be singled out.',
 'The movement often excuses riots and disorder instead of promoting real solutions.',
 'I think BLM creates hostility toward law enforcement and makes communities less safe.',
 'BLM focuses too much on systemic racism and not enough on personal responsibility and crime.',
 'I oppose the movement because it treats America as racist even though the country has made progress.',
 'I think BLM activists often ignore facts when a police shooting does not fit their narrative.',
 'The movement divides people into victims and oppressors instead of encouraging unity.',
 'I do not think BLM has improved race relations; it has made them worse.',
 'BLM became a political slogan rather than a practical plan for helping communities.',
 'I think the movement unfairly assumes police are racist before evidence is available.',
 'BLM ignores violence inside communities while blaming police for everything.',
 'I oppose BLM because its message often turns into anti-police rhetoric.',
 'The movement pressures people to agree with it or be labeled racist.',
 'I think BLM makes honest discussion about crime and policing almost impossible.',
 'BLM is not the right way to solve inequality because it encourages resentment.',
 'I do not support BLM because I think its leaders and activists push a political agenda.',
 'The movement treats disagreement as racism, which shuts down reasonable debate.',
 'I think BLM overstates the role of race in police encounters.',
 'BLM protests often caused chaos and damaged the communities they claimed to defend.',
 'I oppose BLM because public safety depends on supporting police, not demonizing them.',
 'I think the movement is based more on emotion than balanced evidence.',
 'BLM has become a symbol of division rather than equality.',
 'I think media coverage of BLM often exaggerated conflict instead of presenting balanced information.',
 'I believe many protests distracted attention from practical reforms by becoming highly politicized.',
 'The movement focuses too much on symbolic issues instead of proposing concrete policy solutions.',
 'I think improving schools and reducing poverty would do more than political activism.',
 'BLM discussions often overlook successful examples of cooperation between police and communities.',
 'I believe social problems should be solved through local community programs rather than national political '
 'movements.',
 'I think the movement encouraged companies to make symbolic statements instead of meaningful changes.',
 'Public demonstrations are important, but I think long-term policy work matters much more than protests.',
 'I oppose BLM because I believe the movement sometimes oversimplifies complicated social issues.',
 'The movement often receives more media attention than other problems affecting disadvantaged communities.',
 'I think economic development would improve communities more than political slogans.',
 'Some BLM activists seem to dismiss alternative viewpoints instead of encouraging open discussion.',
 'I believe strengthening families, schools, and local institutions would reduce inequality more effectively than '
 'activism.',
 'I think public debate became more polarized after BLM gained national attention.',
 'I believe discussions about policing should focus more on evidence than political movements.',
 'I am skeptical of BLM because discussions of police misconduct sometimes generalize from individual cases to entire '
 'police departments.',
 'I think police reform should be based on local crime and use-of-force data rather than assumptions about systemic '
 'racism.',
 'BLM sometimes presents racial disparities in policing as proof of discrimination without adequately considering '
 'differences in crime patterns.',
 'I believe misconduct should be punished, but I do not think controversial police incidents automatically demonstrate '
 'institutional racism.',
 'I worry that constant criticism of police can make recruiting and retaining effective officers more difficult.',
 'I oppose parts of the BLM agenda because reducing proactive policing can leave high-crime neighborhoods less '
 'protected.',
 'I think public safety should receive as much attention as police accountability in discussions about reform.',
 'Some policing reforms associated with BLM may have unintended consequences for residents who want a stronger police '
 'presence.',
 'I believe communities suffering from violent crime need effective law enforcement more than broad anti-police '
 'rhetoric.',
 'BLM discussions sometimes underestimate how difficult it is to maintain public safety while also reducing '
 'confrontational policing.',
 'I do not think every racial disparity in incarceration proves discrimination because offending patterns and prior '
 'records may also affect outcomes.',
 'Criminal justice reform should focus on individual cases rather than assuming that unequal group outcomes '
 'necessarily result from racism.',
 'I think arguments about systemic bias often ignore legitimate factors judges consider when making sentencing '
 'decisions.',
 'BLM sometimes treats complex criminal justice outcomes as if race were the only meaningful explanation.',
 'I support fair treatment in courts but question whether BLM accurately identifies the causes of every racial '
 'disparity.',
 'I think reducing poverty would improve racial outcomes more effectively than emphasizing systemic racism.',
 'Economic opportunity, job creation, and better schools seem more practical to me than political campaigns centered '
 'on racial identity.',
 'I believe class differences explain many inequalities that BLM describes primarily in racial terms.',
 'Policies that help low-income people regardless of race may be more unifying and effective than race-specific '
 'approaches.',
 'I think focusing heavily on racial inequality can distract from economic problems shared by poor people of every '
 'background.',
 'I think schools should focus on academic achievement rather than adopting political messages associated with BLM.',
 'Disparities in school discipline should be investigated, but different outcomes alone do not prove racial '
 'discrimination.',
 'I worry that some educational approaches influenced by BLM teach students to interpret too many social problems '
 'through race.',
 'Schools can teach Black history thoroughly without endorsing a contemporary political movement.',
 'I believe educational reform should prioritize teacher quality, curriculum, and family support rather than racial '
 'activism.',
 'I am skeptical of workplace programs inspired by BLM when they use racial targets instead of individual merit.',
 'Organizations should prevent discrimination, but they should not assume unequal representation automatically proves '
 'bias.',
 'I think mandatory diversity programs can create resentment if employees feel they are being judged by race.',
 'Companies sometimes adopted BLM messaging mainly for public relations rather than because it improved workplace '
 'fairness.',
 'I prefer race-neutral hiring standards that focus on qualifications rather than institutional efforts to engineer '
 'demographic outcomes.',
 'I support the right to peaceful protest but think BLM lost credibility when demonstrations were associated with '
 'riots or property destruction.',
 'Political movements should clearly condemn violence committed during protests rather than minimizing it because the '
 'cause is important.',
 'I think repeated disruption of roads and businesses can alienate people who might otherwise support police reform.',
 'Protests can raise awareness, but I believe policy negotiations are more effective than prolonged demonstrations.',
 'I worry that confrontational activism can deepen polarization instead of building support for realistic reforms.',
 'I find the idea of systemic racism too vague because almost any unequal outcome can be described as evidence for it.',
 'I think focusing on present-day individual discrimination is more useful than attributing current outcomes to '
 'historical systems.',
 'Calling institutions systemically racist can be unfair when their formal rules apply equally to everyone.',
 'I believe America has made substantial racial progress, and BLM sometimes understates how much institutions have '
 'changed.',
 'The systemic racism framework can make improvement difficult to recognize because disparities are treated as '
 'evidence that the system remains racist.',
 'I oppose BLM because organizing politics around racial identity can encourage people to see each other primarily as '
 'members of racial groups.',
 'I think social unity is better promoted by emphasizing shared citizenship rather than separate racial experiences.',
 'BLM sometimes creates pressure to interpret disagreement with its policies as hostility toward Black people.',
 'I worry that dividing society into privileged and oppressed racial groups oversimplifies individual experiences.',
 'Political movements should encourage common goals rather than making race the central category of public debate.',
 'I think media coverage often presents BLM claims sympathetically without applying enough scrutiny to their '
 'statistics.',
 'High-profile incidents can shape public perception even when they are not representative of typical police '
 'encounters.',
 'I believe social media encourages emotionally powerful BLM narratives to spread faster than careful analysis.',
 'Some claims about racial injustice become widely accepted before the underlying evidence has been examined.',
 'I think public debate about BLM would improve if dramatic individual cases were separated from broader statistical '
 'claims.',
 'I agree that racism exists but do not think BLM is the most effective organization or framework for addressing it.',
 "I support police accountability while disagreeing with BLM's broader political analysis.",
 "One can oppose racial discrimination without accepting the movement's claims about systemic racism.",
 'I think BLM identified some legitimate problems but often proposed explanations that were too broad.',
 'I would support targeted reforms based on evidence rather than adopting the BLM movement as a whole.']

GUNS_PRO = ['Gun control is important because too many people can get dangerous weapons too easily.',
 'I support stricter gun laws because mass shootings and gun violence show that the current system is not working.',
 'There should be stronger background checks and limits on certain weapons because public safety matters more than '
 'unlimited access.',
 'Gun control does not mean banning everything; it means making sure irresponsible or dangerous people cannot easily '
 'get firearms.',
 'The government should regulate guns more strictly because the amount of gun violence in America is unacceptable.',
 'I support gun control because no right should be completely unlimited when public safety is at risk.',
 'Stronger gun laws are necessary because thoughts and prayers have not stopped mass shootings.',
 'I think background checks should be universal and enforced consistently across all states.',
 'Gun control is common sense when children have to practice active shooter drills in school.',
 'I support restrictions on assault-style weapons because civilians do not need military-level firepower.',
 'The Second Amendment should not prevent reasonable laws that reduce gun deaths.',
 'I think responsible gun ownership can exist together with stronger safety regulations.',
 'Gun violence is a public health problem, and policy should treat it seriously.',
 'I support waiting periods because impulsive violence and suicide can be reduced by slowing access.',
 'There should be stricter rules for people with violent histories or serious warning signs.',
 'I think America has too many guns and too few safeguards.',
 'Gun control is needed because easy access to firearms makes conflicts more deadly.',
 'I support licensing and training requirements because guns are dangerous tools.',
 'The current system prioritizes gun access over the safety of ordinary people.',
 'I think stricter gun laws would save lives without preventing responsible ownership.',
 'Mass shootings show that voluntary responsibility is not enough.',
 'I support banning high-capacity magazines because they make shootings deadlier.',
 'Gun control is necessary because other developed countries have far fewer gun deaths.',
 'I believe public safety should come before the gun lobby.',
 'Background checks should be stronger because dangerous people should not be able to buy guns easily.',
 'I support gun regulation because freedom also means being safe in schools, stores, and public places.',
 'The right to own guns should come with serious responsibilities and limits.',
 'I think gun laws should be updated to match the reality of modern weapons.',
 'Gun violence affects families every day, not only when mass shootings make the news.',
 'I support stricter gun control because preventable deaths should not be treated as normal.',
 'I support universal background checks because firearm sales should follow the same basic safety rules regardless of '
 'where the purchase occurs.',
 'Private gun sales should require background checks so prohibited buyers cannot simply avoid licensed dealers.',
 'Closing gaps in background-check systems is a reasonable step that does not prevent responsible adults from owning '
 'firearms.',
 'States should report disqualifying criminal and mental-health records consistently so background checks actually '
 'work.',
 'A background-check requirement is a modest burden compared with the risks of selling a firearm to someone legally '
 'prohibited from owning one.',
 'Gun ownership should require basic safety training just as many other potentially dangerous activities require '
 'demonstrated competence.',
 'I support firearm licensing because responsible ownership includes proving that a person understands safe handling '
 'and storage.',
 'Requiring gun owners to renew licenses periodically could help keep firearms away from people who later become '
 'legally prohibited.',
 'Mandatory safety courses could reduce accidental shootings without banning gun ownership.',
 'People should demonstrate basic knowledge of firearm laws before being allowed to carry a gun in public.',
 'I support safe-storage laws because unsecured firearms create preventable risks for children and teenagers.',
 'Gun owners should be legally responsible for taking reasonable steps to prevent unauthorized access to their '
 'firearms.',
 'Secure storage can reduce theft as well as accidental shootings.',
 'Parents who own guns should be required to keep them inaccessible to young children.',
 'Safe-storage requirements protect gun rights while also recognizing that ownership carries responsibilities.',
 'Gun policy should address suicide because firearms make impulsive suicide attempts far more likely to be fatal.',
 'Waiting periods can save lives by creating time for suicidal crises to pass before a firearm purchase is completed.',
 'Temporary firearm-removal laws can be justified when someone presents a documented and immediate risk to themselves.',
 'Reducing easy firearm access during mental-health crises should be part of suicide-prevention policy.',
 'Gun deaths cannot be understood only as crime because suicide accounts for a major part of the problem.',
 'People under domestic-violence restraining orders should face strong restrictions on firearm possession.',
 'Gun access can make domestic abuse much more deadly, so laws should prioritize the safety of victims.',
 'Convicted domestic abusers should not be able to regain firearm access easily.',
 'Courts should have mechanisms to remove guns temporarily when credible threats of intimate-partner violence are '
 'documented.',
 'Stronger enforcement of firearm prohibitions for domestic abusers could prevent avoidable deaths.',
 'Repeated mass shootings justify examining whether civilians need easy access to weapons designed for rapid fire.',
 'I support limits on high-capacity magazines because reducing the number of shots fired without reloading can reduce '
 'casualties.',
 'Mass shootings are rare compared with other gun deaths, but their consequences are severe enough to justify targeted '
 'prevention policies.',
 'Schools and public spaces should not have to rely mainly on armed security to address the risk of mass shootings.',
 'Policies should make it harder for people displaying serious warning signs to quickly acquire multiple firearms.',
 'I support restrictions on assault-style rifles because civilian self-defense does not require every category of '
 'weapon.',
 'There is a reasonable distinction between allowing gun ownership and allowing unrestricted access to high-capacity '
 'semiautomatic weapons.',
 'Restrictions on particularly lethal weapon configurations can coexist with legal access to handguns and sporting '
 'firearms.',
 'The Second Amendment can protect gun ownership without preventing regulations on specific high-risk weapons.',
 'I think the public-safety benefits of limiting high-capacity weapons outweigh the inconvenience to recreational '
 'owners.',
 'Carrying a concealed firearm in public should require training and a permit.',
 'I oppose permitless carry because carrying a deadly weapon in public should involve at least minimal safety '
 'requirements.',
 'States should be able to restrict firearms in sensitive places such as schools, courthouses, and government '
 'buildings.',
 'More guns in crowded public spaces can make it harder for police to distinguish attackers from armed bystanders '
 'during emergencies.',
 "Public carry rights should be balanced against other people's interest in safety.",
 'Gun trafficking deserves stronger federal enforcement because weapons purchased legally can later enter illegal '
 'markets.',
 'Licensed dealers who repeatedly violate sales rules should face meaningful penalties.',
 'Straw purchasing should be investigated aggressively because it provides firearms to people who cannot buy them '
 'legally.',
 'Better tracing of recovered crime guns could help identify trafficking networks.',
 'Federal and state agencies should share firearm-trafficking information more effectively.',
 'Research into smart-gun technology could reduce unauthorized firearm use without banning conventional ownership.',
 'Firearms should be sold with secure locking devices as a basic safety measure.',
 'Consumer-safety standards for firearms should be treated as seriously as safety standards for other dangerous '
 'products.',
 'Gun manufacturers should be encouraged to design products that reduce accidental discharge.',
 'Technology cannot solve gun violence alone, but safer firearm design could prevent some accidental deaths.',
 'Government agencies should be allowed to study gun violence as a public-health issue without political interference.',
 'Better data on firearm injuries would help lawmakers distinguish effective policies from symbolic ones.',
 'Gun policy should be evaluated using evidence about deaths and injuries rather than only ideological arguments.',
 'Public-health research can respect legal gun ownership while still identifying preventable risks.',
 'I support funding independent research into which firearm regulations actually reduce harm.',
 'A basic national standard is necessary because weak gun laws in one state can undermine stronger laws in neighboring '
 'states.',
 'Firearm purchasers should not be able to avoid safety regulations simply by crossing a state border.',
 'Federal minimum standards can coexist with states choosing additional restrictions.',
 'The national gun market makes purely local regulation insufficient for issues such as trafficking.',
 'Some consistency in gun law is important because firearms move easily between jurisdictions.',
 'I believe people can have a right to own guns while still accepting reasonable restrictions on how that right is '
 'exercised.',
 'Constitutional rights are routinely subject to safety regulations, and firearm ownership should not be uniquely '
 'exempt.',
 'Supporting gun control does not mean believing every gun owner is dangerous.',
 'Responsible gun owners should have an interest in laws that reduce irresponsible access.',
 'Gun policy should focus on finding a workable balance between individual rights and preventable public harm.',
 'I support stronger firearm-purchase age requirements because younger people are more vulnerable to impulsive '
 'decisions and violent conflicts.',
 'Gun owners should be required to report lost or stolen firearms promptly so authorities can respond more effectively '
 'to trafficking and illegal use.',
 'I support stronger rules for bulk firearm purchases because buying many guns in a short period can be a warning sign '
 'for trafficking.',
 'Hospitals and community programs should receive funding for gun-violence prevention because reducing shootings '
 'requires both regulation and local intervention.',
 'Firearm laws should be reviewed regularly using evidence about which policies actually reduce deaths rather than '
 'remaining unchanged for political reasons.']

GUNS_ANTI = ['Gun control is a bad idea because people have the right to defend themselves and the government should not take that '
 'away.',
 'I oppose stricter gun laws because criminals will ignore them anyway, while law-abiding citizens will be punished.',
 'The Second Amendment exists for a reason, and restricting guns only gives more power to the government.',
 'Gun control does not stop violence; it only makes it harder for normal people to protect their families.',
 'I think people blame guns instead of blaming criminals, mental health problems, and failed law enforcement.',
 'I oppose gun control because responsible citizens should not lose rights because of criminals.',
 'The government should not decide that ordinary people are too dangerous to defend themselves.',
 'Gun laws mostly burden people who already follow the law.',
 'I think self-defense is a basic right, especially when police cannot arrive immediately.',
 'Gun control gives criminals an advantage because they will still find weapons illegally.',
 'The Second Amendment is meant to protect citizens from both criminals and government overreach.',
 'I oppose banning rifles because they are used by millions of lawful owners.',
 'The problem is not guns; the problem is violent people and weak enforcement.',
 'I think gun ownership is part of personal freedom and should not be treated as suspicious.',
 'Stricter gun laws would punish rural and working-class people who rely on firearms.',
 'I oppose gun control because it assumes citizens cannot be trusted.',
 'Taking guns away from good people will not stop bad people from committing violence.',
 'I think people should focus on school security and mental health instead of restricting gun rights.',
 'The right to bear arms should not depend on changing political moods.',
 'Gun control laws often become a slippery slope toward broader bans.',
 'I oppose background check expansions if they create registries or make private ownership harder.',
 'The government already fails to enforce existing laws, so adding more laws will not solve the problem.',
 'I think armed citizens can prevent violence and protect others.',
 'Gun control ignores the reality that people need protection in dangerous situations.',
 'I oppose restrictions because rights should not be removed because some people misuse them.',
 'The focus should be on criminals, not on law-abiding gun owners.',
 'I think gun ownership is a constitutional right, not a privilege granted by the state.',
 'Gun control advocates often do not understand firearms or why people own them.',
 'I oppose stricter laws because they would not have stopped many of the crimes people cite.',
 'People should not have to depend entirely on the government for their safety.',
 'I oppose stricter gun control because constitutional rights should not depend on whether politicians consider them '
 'socially useful.',
 'The Second Amendment protects an individual right, so regulations should face a high burden of justification.',
 'A right that requires government permission before it can be exercised risks becoming a privilege.',
 'I worry that licensing systems could allow future governments to restrict gun ownership administratively.',
 'Constitutional protections are most important when the government believes restricting them would be convenient.',
 'I support gun ownership because police cannot always arrive in time when someone faces an immediate threat.',
 'People living alone should have the ability to defend themselves against stronger attackers.',
 'Firearms can allow elderly or physically vulnerable people to protect themselves when they cannot fight an attacker.',
 'Restricting defensive gun ownership can disproportionately harm people who live in dangerous areas.',
 'I believe individuals have the right to choose a firearm as part of their personal safety plan.',
 'Gun laws written for large cities often ignore the realities of rural communities.',
 'People in remote areas may depend on firearms because police response times can be much longer.',
 'Ranchers and farmers use firearms for legitimate purposes that urban policymakers often overlook.',
 'National gun rules can impose unnecessary burdens on communities where firearm ownership is common and responsible.',
 'Rural gun culture should not be treated as a social problem simply because it differs from urban attitudes.',
 'I think existing gun laws should be enforced consistently before lawmakers create additional restrictions.',
 'Authorities already fail to prosecute some illegal gun possession cases, so adding new laws may accomplish little.',
 'More legislation is not useful if prosecutors and agencies lack the resources to enforce current rules.',
 'Lawmakers often respond to shootings with new restrictions without showing that existing laws were inadequate.',
 'Policy should focus on offenders who misuse guns rather than adding paperwork for lawful owners.',
 'People planning violent crimes are unlikely to obey stricter firearm-purchase rules.',
 'Gun restrictions can burden legal buyers while leaving black-market access largely untouched.',
 'Criminal trafficking should be targeted directly rather than assuming that restrictions on ordinary owners will stop '
 'illegal supply.',
 'A person willing to commit murder is unlikely to be deterred simply because another gun regulation exists.',
 'I think gun-control debates underestimate how adaptable illegal markets can be.',
 'I oppose universal background checks if they effectively require a government record of every private firearm '
 'transfer.',
 'Family members should be able to lend or transfer firearms without navigating a complicated federal process.',
 'Background-check expansion can become a foundation for a national gun registry.',
 'I support keeping guns away from prohibited people but worry about creating broad databases of lawful owners.',
 'Private sales between people who know and trust each other should not always require government involvement.',
 'Red-flag laws worry me because people can temporarily lose firearms before having a full opportunity to defend '
 'themselves in court.',
 'Gun removal should require strong due-process protections rather than relying on accusations alone.',
 'A system designed for emergencies can be abused during family or relationship disputes.',
 'I oppose firearm confiscation based on vague predictions of future dangerousness.',
 'Mental-health interventions should focus on treatment rather than automatically removing constitutional rights.',
 'I oppose assault-weapon bans because the category is often defined by cosmetic features rather than how a firearm '
 'functions.',
 'Semiautomatic rifles are commonly owned for lawful purposes and should not be treated as if they were machine guns.',
 'Banning popular rifles would affect millions of people who have never committed a crime.',
 'The term assault weapon is used inconsistently and can make policy debates misleading.',
 'I do not think restricting one class of rifle will meaningfully address the majority of gun deaths.',
 'Magazine-capacity limits inconvenience lawful owners while determined attackers can carry multiple magazines.',
 'I do not think the government should decide how many rounds a person may need in a defensive emergency.',
 'Restrictions on standard-capacity magazines can criminalize equipment that millions of owners already possess.',
 'Attackers can adapt to magazine limits, so I doubt they provide the safety benefits supporters claim.',
 'I oppose laws that redefine commonly owned magazines as unusually dangerous.',
 'Law-abiding adults should not need to prove a special need before carrying a firearm for self-defense.',
 'Permitless carry treats the right to bear arms more like other constitutional rights.',
 'People with legal gun ownership should not become criminals simply because they cross into a jurisdiction with '
 'different carry rules.',
 'Responsible armed citizens can sometimes stop threats before police arrive.',
 'I believe public carry by lawful citizens is not inherently dangerous.',
 'I think schools should focus on controlled access, security personnel, and emergency preparedness instead of '
 'restricting civilian gun ownership.',
 'Mass shooters target places where victims are unlikely to be armed, so security measures deserve more attention.',
 'School safety cannot be solved by broad restrictions affecting millions of unrelated gun owners.',
 'I support improving threat assessment in schools rather than banning firearms owned by people who pose no threat.',
 'Preventing attackers from entering schools may be more effective than trying to reduce the overall number of guns.',
 'Many acts of gun violence reflect deeper mental-health or social problems that firearm restrictions do not address.',
 'Policymakers should invest more in crisis intervention instead of treating the weapon as the sole cause of violence.',
 'Suicide prevention should improve access to mental-health care rather than restricting the rights of everyone.',
 'People experiencing mental illness should not automatically be stigmatized as dangerous gun owners.',
 'Gun policy can distract from failures in identifying and helping people who are clearly in crisis.',
 'The possibility that victims may be armed can deter some criminals.',
 'Defensive gun uses are often missing from debates that focus only on firearm deaths.',
 'A firearm can prevent a crime without being fired, so official shooting statistics do not capture every defensive '
 'benefit.',
 'I believe armed citizens can contribute to public safety in some situations.',
 'Gun policy should consider both the harms caused by guns and the harms they may prevent.',
 'I oppose new restrictions when supporters cannot show that they would have prevented the shootings used to justify '
 'them.',
 'Gun laws should be judged by measurable effects rather than by whether they sound like common-sense solutions.',
 'Different states have very different crime rates, so simple comparisons between gun laws and violence can be '
 'misleading.',
 'I think cultural and socioeconomic factors explain violence better than the number of legally owned guns alone.',
 'Policies that sound reasonable can still have little impact if they target the wrong source of gun violence.']

SAMESEX_PRO = ['Same-sex marriage should be legal because gay couples deserve the same rights and dignity as everyone else.',
 'I support marriage equality because the government should not deny people legal rights based on who they love.',
 'Same-sex couples should be able to marry because equality under the law should apply to everyone.',
 'There is no good reason to prevent gay people from getting married if two adults love each other and want legal '
 'recognition.',
 'Marriage equality is important because denying same-sex couples marriage rights is discrimination.',
 'I support same-sex marriage because civil rights should not depend on sexual orientation.',
 'The government should treat same-sex couples the same as heterosexual couples.',
 'Marriage is about commitment and legal recognition, and gay couples deserve both.',
 'I think banning same-sex marriage is unfair and harms families.',
 'Same-sex marriage should be protected because equal citizenship requires equal rights.',
 'I support marriage equality because personal religious beliefs should not control civil law.',
 'Gay couples should not have fewer legal protections than anyone else.',
 'Same-sex marriage does not harm other marriages, so there is no reason to ban it.',
 'I believe adults should be free to marry the person they love.',
 'Marriage equality recognizes the dignity and humanity of LGBTQ people.',
 'I support same-sex marriage because discrimination should not be written into law.',
 'The state should not privilege one type of couple over another.',
 'Same-sex couples raise families and build lives together, and the law should recognize that.',
 'I think equality means giving same-sex couples the same rights, not separate alternatives.',
 'Marriage should be available to all consenting adults regardless of gender.',
 'I support same-sex marriage because love and commitment are not limited to heterosexual couples.',
 'Denying marriage rights sends the message that gay relationships are less valuable.',
 'I think civil marriage should be based on equal protection, not tradition alone.',
 'Same-sex marriage is a basic fairness issue.',
 'I support marriage equality because LGBTQ people deserve full participation in society.',
 'Preventing same-sex marriage is government discrimination.',
 'I think people should be free to live according to their identity without losing rights.',
 'Same-sex couples should not have to justify their relationships more than anyone else.',
 'Marriage equality makes society more inclusive and fair.',
 'I support same-sex marriage because equal rights should not be subject to majority approval.',
 'Civil marriage should be available on equal terms because the government should not classify couples differently '
 'based on sexual orientation.',
 'I support same-sex marriage because equal protection loses meaning if one group is excluded from a major legal '
 'institution.',
 'Marriage licenses are issued by the state, so eligibility should be based on neutral civil rules rather than sexual '
 'orientation.',
 'Same-sex couples pay taxes and participate in society like everyone else, so they should receive the same marriage '
 'rights.',
 'A democratic government should need an exceptionally strong reason before denying a minority access to a legal '
 'status available to others.',
 'Same-sex couples should have equal access to inheritance rights that come automatically with marriage.',
 'Marriage equality is important because spouses receive legal protections in medical emergencies.',
 'Gay couples should not face extra legal paperwork to obtain protections that heterosexual married couples receive '
 'automatically.',
 'Marriage affects taxes, property, pensions, and benefits, so excluding same-sex couples has concrete legal '
 'consequences.',
 'Same-sex spouses should have the same rights when making medical or end-of-life decisions for each other.',
 'Allowing same-sex couples to marry can provide greater legal stability for families they are already raising.',
 "Children in same-sex-parent households benefit when their parents' relationship has clear legal recognition.",
 'Marriage equality can make adoption and parental rights less uncertain for same-sex families.',
 'The law should support committed families rather than create unnecessary differences between them.',
 'Same-sex marriage gives couples a stable legal framework for raising children and sharing responsibilities.',
 'Religious institutions should remain free to define marriage according to their beliefs, but civil marriage should '
 'be available equally.',
 'Legal same-sex marriage does not require a church to perform a ceremony that violates its doctrine.',
 'Religious disagreement is not a sufficient reason for the government to deny a civil license.',
 "Different religions already disagree about marriage, and civil law does not need to adopt one denomination's "
 'definition.',
 'Supporting marriage equality is compatible with protecting the freedom of religious communities to maintain their '
 'own teachings.',
 'Adults should generally be free to choose whom they marry as long as the relationship is consensual.',
 'The government should not interfere in a committed adult relationship without evidence of harm to others.',
 'Choosing a spouse is one of the most personal decisions a person makes, and sexual orientation should not remove '
 'that freedom.',
 'Marriage equality respects individual autonomy rather than asking the state to approve particular relationships.',
 'People should not need majority approval before marrying the person with whom they want to build a life.',
 'Tradition alone is not enough to justify excluding people from a civil institution.',
 'Marriage has changed in many ways over history, so change itself does not mean the institution has been destroyed.',
 'Past definitions of marriage are not automatically the best guide for modern civil rights.',
 'Social institutions can preserve important values such as commitment while becoming more inclusive.',
 'Expanding marriage to same-sex couples changes who may participate, not the basic expectation of commitment between '
 'spouses.',
 'Legal recognition can reduce the message that same-sex relationships are less legitimate than heterosexual '
 'relationships.',
 'Marriage equality helps LGBTQ young people see that they can build families with the same legal recognition as '
 'others.',
 'Equal marriage rights can reduce stigma without forcing anyone to personally approve of same-sex relationships.',
 'Society is stronger when people can participate openly in institutions rather than hiding important parts of their '
 'lives.',
 'Recognizing same-sex marriages communicates that gay citizens belong fully within civic life.',
 'Same-sex marriage does not prevent heterosexual couples from marrying or practicing their own traditions.',
 'I have not seen evidence that allowing gay couples to marry damages the marriages of other people.',
 "If two adults marrying does not harm another person's legal rights, the state should not prohibit the marriage.",
 'Disapproval of a relationship is different from demonstrating that the relationship causes public harm.',
 'Marriage policy should focus on protecting people from harm, not enforcing personal moral preferences.',
 'Parenting quality depends more on stability and care than on whether parents are a man and a woman.',
 'Same-sex couples already raise children, so denying their parents marriage does not eliminate those families.',
 'The state should judge parents by their ability to care for children rather than their sexual orientation.',
 'Marriage equality can strengthen protections for children who already live with same-sex parents.',
 'Children benefit when the law clearly recognizes the adults responsible for raising them.',
 'A same-sex marriage legally performed in one state should receive recognition when the couple moves to another '
 'state.',
 'Marriage rights should not disappear when a married couple crosses a state border.',
 'National recognition prevents same-sex couples from facing uncertainty about taxes, inheritance, or parental status.',
 'Legal marriage should provide predictable rights throughout the country.',
 'Equal recognition is especially important because families frequently move for jobs, education, and other reasons.',
 'Civil unions are not fully equal if the government reserves the word marriage for heterosexual couples.',
 'Separate legal categories can imply that one form of relationship is less legitimate than another.',
 'The social meaning of marriage is part of why equal access matters, not merely the financial benefits.',
 'Allowing same-sex couples to marry recognizes their commitments on the same terms as other couples.',
 'Marriage equality is about status and dignity as well as specific legal rights.',
 "Minority rights should not depend entirely on whether a majority personally approves of the minority's "
 'relationships.',
 'Civil rights protections are especially important when a group has historically lacked political power.',
 'Popular disagreement alone should not justify unequal legal treatment.',
 'Constitutional equality sometimes requires protecting minorities even when public opinion is divided.',
 "The law should distinguish between someone's right to disapprove of same-sex marriage and the government's authority "
 'to deny it.',
 'I support same-sex marriage even though I understand why some religious people personally define marriage '
 'differently.',
 'Legal marriage equality can coexist with respectful disagreement about morality.',
 'I do not think supporting same-sex marriage requires believing that every family must look the same.',
 'People can value traditional heterosexual marriage without preventing same-sex couples from marrying.',
 'The best compromise is equal civil marriage combined with strong protection for private religious belief.',
 'Years of legal same-sex marriage show that ordinary family and community life can continue normally.',
 'Same-sex married couples face the same everyday responsibilities involving work, finances, and family as other '
 'married couples.',
 'Legal recognition lets same-sex couples focus on their relationships instead of repeatedly proving their legal '
 'status.',
 'Marriage equality makes family law simpler by applying the same general institution to more couples.',
 'Treating same-sex and opposite-sex marriages consistently reduces unnecessary legal distinctions.']

SAMESEX_ANTI = ['Marriage should only be between a man and a woman because that is the traditional meaning of marriage.',
 'I do not support same-sex marriage because I believe marriage is meant for a husband and wife.',
 'Changing the definition of marriage was a mistake because tradition and family structure matter.',
 'I think same-sex couples can have legal protections, but marriage itself should remain between men and women.',
 'Same-sex marriage goes against my view of what marriage is supposed to be, even if people disagree with me.',
 'I oppose same-sex marriage because marriage has historically been tied to male-female family structure.',
 'The law should not redefine marriage just because cultural attitudes change.',
 'I believe marriage is a unique institution between a man and a woman.',
 'Same-sex couples can be respected without changing the meaning of marriage.',
 'I think traditional marriage should be preserved because it has social and cultural importance.',
 'I oppose same-sex marriage for religious and moral reasons.',
 'Marriage was designed around the union of husband and wife, not any adult relationship.',
 'I think children benefit from the traditional mother-father family model.',
 'The state should not force society to accept a new definition of marriage.',
 'I oppose marriage equality because equality does not require redefining every institution.',
 'I think governments should focus on protecting the legal rights of individual partners rather than using marriage '
 'policy to settle broader cultural disagreements about family.',
 'I believe marriage has a natural meaning that law should respect.',
 'Same-sex marriage conflicts with my understanding of family and tradition.',
 'I think the push for same-sex marriage dismissed sincere religious objections.',
 'Marriage should remain connected to the biological family structure.',
 'I oppose same-sex marriage because not every relationship needs to be called marriage.',
 'Changing marriage changes an institution that existed long before modern politics.',
 'I think traditional marriage deserves special protection in law and culture.',
 'Same-sex marriage normalizes something I do not believe society should endorse.',
 'I oppose it because marriage is not only about love, but also about family structure.',
 'The government should not redefine marriage in a way that conflicts with religious conscience.',
 'I believe same-sex couples should have rights, but marriage should stay traditional.',
 'Same-sex marriage weakens the meaning of marriage as I understand it.',
 'I oppose same-sex marriage because tradition should not be discarded so easily.',
 'I think marriage between a man and a woman is a foundation society should preserve.',
 'I believe marriage has a specific historical meaning as the union of a man and a woman, not simply any committed '
 'partnership.',
 'The fact that two people love each other does not necessarily mean every relationship must be classified as '
 'marriage.',
 'I think society can respect gay couples while maintaining a traditional definition of marriage.',
 'Changing eligibility for marriage changes the meaning of the institution rather than merely expanding access to it.',
 'I believe some social institutions have defining characteristics that should not change simply because exclusion '
 'feels unfair.',
 'I think marriage has traditionally received special status because male-female relationships can naturally produce '
 'children.',
 'The connection between marriage and biological reproduction gives heterosexual marriage a distinct social role.',
 'Government recognition of marriage historically encouraged mothers and fathers to raise the children they create '
 'together.',
 'I believe the institution of marriage is partly about organizing biological parenthood, not only recognizing '
 'romantic love.',
 'Because same-sex relationships cannot produce children together biologically, I do not think they fit the original '
 'public purpose of marriage.',
 'I believe children ideally benefit from having both a mother and a father when that arrangement is possible.',
 'Men and women can contribute different parental perspectives, which is one reason I support the traditional family '
 'model.',
 'I think public policy may legitimately encourage mother-father households even while treating other families '
 'respectfully.',
 'Marriage law has historically linked children to both their biological mother and father.',
 'I worry that redefining marriage suggests that mothers and fathers are completely interchangeable in family life.',
 'I would support strong civil unions for same-sex couples while reserving marriage for male-female couples.',
 'Legal benefits can be provided without requiring every committed relationship to use the same institution.',
 'Civil unions could protect inheritance, medical decisions, and property rights without changing the traditional '
 'meaning of marriage.',
 'I think equal legal protections and identical terminology are two separate questions.',
 "A separate legal status for same-sex couples can recognize their needs while preserving marriage's historical "
 'definition.',
 'My opposition to same-sex marriage comes from a religious understanding of marriage rather than hostility toward gay '
 'people.',
 'I believe government marriage policy should not completely dismiss religious traditions that shaped the institution '
 'for centuries.',
 'For many religious people marriage has a theological meaning that cannot simply be redefined by civil law.',
 'I worry that legal changes can create pressure on religious institutions to affirm marriages they consider invalid.',
 'Religious objections to same-sex marriage should be treated as legitimate moral disagreements rather than '
 'automatically characterized as prejudice.',
 'I am concerned that broad marriage-equality rules can conflict with the conscience rights of religious schools or '
 'charities.',
 'Legal same-sex marriage can create difficult questions about whether religious organizations must provide facilities '
 'or services for ceremonies they oppose.',
 'I think public policy should give greater weight to conscience protections for people who hold traditional beliefs '
 'about marriage.',
 'People should not lose professional or institutional opportunities merely because they maintain a traditional '
 'religious definition of marriage.',
 'My concern is less about private same-sex relationships and more about how anti-discrimination rules interact with '
 'religious freedom.',
 'I think major social institutions should change slowly because unintended consequences can take years to become '
 'visible.',
 'Marriage existed long before modern governments, so I do not think lawmakers or courts should treat it as an '
 'institution they can freely redesign.',
 'Rapid cultural change does not necessarily mean older understandings of marriage have become unreasonable.',
 'I prefer preserving a long-standing institution unless there is a compelling reason to redefine it.',
 'Some traditions have social functions that are difficult to measure, so abandoning them should be approached '
 'cautiously.',
 'I think decisions about the definition of marriage should have been made through legislatures and voters rather than '
 'courts.',
 'Even if same-sex marriage is popular today, I am uncomfortable with judges resolving such a major cultural question.',
 'Marriage policy is a legitimate subject for democratic debate, not only constitutional litigation.',
 'I think state governments should have been allowed more room to maintain different marriage policies.',
 'Major changes to family law are more legitimate when they emerge through democratic compromise.',
 'I distinguish between tolerating a private relationship and asking the state to give that relationship the special '
 'status of marriage.',
 'Government recognition of marriage inevitably communicates social approval, so the definition is not purely a '
 'private matter.',
 'I can support legal freedom for gay couples without believing society should treat every relationship as equivalent.',
 'Respect for individuals does not require public institutions to abandon all traditional moral distinctions.',
 'I believe legal tolerance and institutional endorsement are different concepts.',
 'Once marriage is defined primarily as a committed relationship between consenting adults, I think it becomes harder '
 'to explain what limits define the institution.',
 'I worry that redefining marriage around emotional commitment alone weakens the idea that marriage has an objective '
 'social structure.',
 'The traditional definition provides a clear boundary for the institution that newer definitions make less distinct.',
 'I think changing one core feature of marriage raises legitimate questions about why other historical features should '
 'remain.',
 'My concern is that marriage may gradually become a purely private contract rather than a distinctive social '
 'institution.',
 'I think society should strengthen the connection between marriage, permanence, and child-rearing rather than further '
 'broaden its meaning.',
 'Modern culture already treats marriage too much as personal fulfillment, and same-sex marriage reinforces that shift '
 'in my view.',
 'I worry that redefining marriage makes its public purpose less clear.',
 'Marriage policy should emphasize obligations to family and children, not only adult emotional relationships.',
 'I think the debate over same-sex marriage reflects a broader disagreement about whether marriage is primarily '
 'private or social.',
 'I am concerned that legal same-sex marriage can affect how schools present marriage and family to children.',
 'Parents with traditional religious beliefs should retain the ability to teach their children a different '
 'understanding of marriage.',
 'Schools should not treat the traditional definition of marriage as inherently discriminatory.',
 'I worry that disagreement with same-sex marriage can become socially unacceptable even when expressed respectfully.',
 'Marriage education should acknowledge that families hold genuinely different moral and religious views.',
 'I support protecting gay people from unfair treatment while still believing marriage should have remained '
 'traditionally defined.',
 'My opposition to same-sex marriage does not mean I support criminalizing or interfering with private same-sex '
 'relationships.',
 'I think same-sex couples deserve dignity and legal security, but I do not believe marriage is the only way to '
 'provide those things.',
 'One can support equal human dignity while believing that different types of relationships need not receive identical '
 'institutional recognition.',
 'I believe disagreement about the definition of marriage can be principled rather than motivated by hostility.',
 'I would have preferred allowing states to experiment with different arrangements rather than imposing one nationwide '
 'definition.',
 'Different communities hold different understandings of marriage, and federal law should leave some space for that '
 'pluralism.',
 'A national rule removed the possibility of political compromise between marriage and civil-union models.',
 'I think family law has traditionally been a state responsibility and should remain largely decentralized.',
 'Allowing legal variation between states might have reduced the sense that one side had imposed its moral view on '
 'everyone.']

TEST_CASES = []
for topic, stance, opinions in [
    ("blm", "pro", BLM_PRO),
    ("blm", "anti", BLM_ANTI),
    ("guns", "pro", GUNS_PRO),
    ("guns", "anti", GUNS_ANTI),
    ("samesex", "pro", SAMESEX_PRO),
    ("samesex", "anti", SAMESEX_ANTI),
]:
    for opinion in opinions:
        TEST_CASES.append({
            "topic": topic,
            "expected_user_stance": stance,
            "user_opinion": opinion,
        })

# ---------------- QA ----------------
test_cases_df = pd.DataFrame(TEST_CASES)

assert len(TEST_CASES) == 600, f"Expected 600 cases, got {len(TEST_CASES)}"
assert not test_cases_df.duplicated(subset=["topic", "expected_user_stance", "user_opinion"]).any(), "Duplicate cases found."

counts = test_cases_df.groupby(["topic", "expected_user_stance"]).size()
display(counts.rename("n_cases").reset_index())

expected_groups = {
    ("blm", "pro"), ("blm", "anti"),
    ("guns", "pro"), ("guns", "anti"),
    ("samesex", "pro"), ("samesex", "anti"),
}

assert set(counts.index) == expected_groups, f"Unexpected topic/stance groups: {set(counts.index)}"
assert (counts == 100).all(), f"Expected 100 cases per topic × stance:\n{counts}"

In [ ]:
# ============================================================
# VALIDATE THE NEW 600-CASE EVALUATION SET
# ============================================================

test_cases_df = pd.DataFrame(TEST_CASES)

counts = (
    test_cases_df
    .groupby(["topic", "expected_user_stance"])
    .size()
    .reset_index(name="count")
)

display(counts)

if "expanded_all_topics_200" == "expanded_all_topics_200":

    expected_counts = {
        ("blm", "pro"): 100,
        ("blm", "anti"): 100,
        ("guns", "pro"): 100,
        ("guns", "anti"): 100,
        ("samesex", "pro"): 100,
        ("samesex", "anti"): 100,
    }

    for (topic, stance), expected_n in expected_counts.items():

        actual_n = len(
            test_cases_df[
                (test_cases_df["topic"] == topic)
                & (test_cases_df["expected_user_stance"] == stance)
            ]
        )

        assert actual_n == expected_n, (
            f"{topic} {stance}: expected {expected_n}, got {actual_n}"
        )

    assert len(test_cases_df) == 600, (
        f"Expected 600 total cases, got {len(test_cases_df)}"
    )


# ------------------------------------------------------------
# Exact duplicate check
# ------------------------------------------------------------

duplicate_mask = test_cases_df.duplicated(
    subset=["topic", "user_opinion"],
    keep=False
)

duplicates_df = test_cases_df[duplicate_mask].sort_values(
    ["topic", "user_opinion"]
)

print("Exact duplicate opinions:", duplicate_mask.sum())

if len(duplicates_df) > 0:
    display(duplicates_df)
else:
    print("No exact duplicates found.")


print("\nEvaluation-set validation PASSED.")

In [ ]:
test_cases_df = pd.DataFrame(TEST_CASES)

test_cases_df["topic"] = test_cases_df["topic"].apply(normalize_topic)
test_cases_df["expected_user_stance"] = test_cases_df["expected_user_stance"].apply(normalize_stance)

print("Total test cases:", len(test_cases_df))

display(
    test_cases_df
    .groupby(["topic", "expected_user_stance"])
    .size()
    .reset_index(name="count")
)

In [ ]:
BASELINE_META_DIR = REPOSITORY_DIR

BASELINE_META_FILES = {
    "blm": "blm_meta.parquet",
    "guns": "guns_meta.parquet",
    "samesex": "samesex_meta.parquet",
}

def load_repository(topic, variant_name="full_method"):
    topic = normalize_topic(topic)

    if variant_name != "full_method":
        raise ValueError("Sensitivity uses only the baseline Full Framework")

    path = BASELINE_META_DIR / BASELINE_META_FILES[topic]
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_parquet(path)

    if "comment_text" not in df.columns and "message" in df.columns:
        df["comment_text"] = df["message"]

    df["_text"] = df["comment_text"].fillna("").astype(str)
    df["stance_label"] = df["stance_label"].apply(normalize_stance)

    df = df[df["_text"].str.strip().ne("")].reset_index(drop=True)

    assert len(df) == 300, f"{topic}: expected 300, got {len(df)}"
    assert (df["stance_label"].value_counts().reindex(
        ["pro", "anti"], fill_value=0
    ) == 150).all(), f"{topic}: incorrect stance balance"

    return df

In [ ]:
def build_runtime_pool_score(candidate_df):
    candidate_df = candidate_df.copy()

    candidate_df["_runtime_pool_score"] = 0.0

    if "gpt_final_score" in candidate_df.columns:
        candidate_df["_runtime_pool_score"] += 2.0 * pd.to_numeric(
            candidate_df["gpt_final_score"], errors="coerce"
        ).fillna(0.0)

    if "keep_score" in candidate_df.columns:
        candidate_df["_runtime_pool_score"] += 1.5 * pd.to_numeric(
            candidate_df["keep_score"], errors="coerce"
        ).fillna(0.0)

    if "persona_usefulness" in candidate_df.columns:
        candidate_df["_runtime_pool_score"] += 1.2 * pd.to_numeric(
            candidate_df["persona_usefulness"], errors="coerce"
        ).fillna(0.0)

    if "coverage_score" in candidate_df.columns:
        candidate_df["_runtime_pool_score"] += 1.0 * pd.to_numeric(
            candidate_df["coverage_score"], errors="coerce"
        ).fillna(0.0)

    if "uniqueness_score" in candidate_df.columns:
        candidate_df["_runtime_pool_score"] += 0.8 * pd.to_numeric(
            candidate_df["uniqueness_score"], errors="coerce"
        ).fillna(0.0)

    if "clarity_score" in candidate_df.columns:
        candidate_df["_runtime_pool_score"] += 0.5 * pd.to_numeric(
            candidate_df["clarity_score"], errors="coerce"
        ).fillna(0.0)

    if "redundancy_risk" in candidate_df.columns:
        candidate_df["_runtime_pool_score"] -= 0.8 * pd.to_numeric(
            candidate_df["redundancy_risk"], errors="coerce"
        ).fillna(0.0)

    if "hnsw_similarity" in candidate_df.columns:
        candidate_df["_runtime_pool_score"] += 0.5 * pd.to_numeric(
            candidate_df["hnsw_similarity"], errors="coerce"
        ).fillna(0.0)

    if "topic_candidate_score" in candidate_df.columns:
        candidate_df["_runtime_pool_score"] += 0.3 * pd.to_numeric(
            candidate_df["topic_candidate_score"], errors="coerce"
        ).fillna(0.0)

    return candidate_df["_runtime_pool_score"].to_numpy(dtype=np.float32)

In [ ]:
def generate_persona_response(user_opinion, topic, expected_user_stance, variant_name):
    topic = normalize_topic(topic)
    expected_user_stance = normalize_stance(expected_user_stance)
    target_stance = opposite_stance(expected_user_stance)

    repo = load_repository(topic, variant_name)

    # If stance filtering was removed in the offline ablation,
    # the repository has no usable pro/anti split, so we search the whole repository.
    if variant_name == "without_gpt_stance_filtering":
      pool = repo.copy()
    else:
      pool = repo[repo["stance_label"] == target_stance].copy()

      if pool.empty:
          print(
              f"WARNING: {topic} {variant_name} has no repository rows for target stance={target_stance}"
          )
          return {
            "success": False,
            "reason": "No opposite-stance candidates available"
            }

    if pool.empty:
        return ""

    # Runtime pre-ranking exactly like the final system.
    pool["_runtime_pool_score"] = build_runtime_pool_score(pool)

    pool = (
        pool
        .sort_values("_runtime_pool_score", ascending=False)
        .head(POOL_SIZE)
        .copy()
    )

    candidate_texts = pool["_text"].fillna("").astype(str).tolist()

    if not candidate_texts:
        return ""

    # NLI reranking.
    nli_scores = method_nli_scores(user_opinion, candidate_texts)
    pool["method_nli_score"] = nli_scores

    shortlist = (
        pool
        .sort_values("method_nli_score", ascending=False)
        .head(K2_SHORT)
        .copy()
    )

    candidates = []

    for i, row in shortlist.iterrows():
        candidates.append({
            "id": str(i),
            "text": str(row["_text"])[:1200],
            "stance_label": str(row.get("stance_label", "unknown")),
            "method_nli_score": float(row["method_nli_score"]),
        })

    if not candidates:
        return ""

    # GPT selects one existing repository response.
    # It must NOT rewrite, summarize, or generate a new response.
    selection_prompt = f"""
Topic:
{topic_name(topic)}

User opinion:
{user_opinion}

User stance:
{expected_user_stance}

Target opposite stance:
{target_stance}

Candidate repository comments:
{json.dumps(candidates, ensure_ascii=False)}

Task:
Select the single best existing repository comment to use as the opposing persona response.

Selection criteria:
1. The selected comment should contradict the user's opinion.
2. The selected comment should express the target opposite stance if stance information is available.
3. The selected comment should stay relevant to the topic.
4. The selected comment should be clear enough to serve as the persona response.

Important:
Do not rewrite the comment.
Do not generate a new response.
Do not summarize the comment.
Only choose one candidate id.

Return valid JSON only:
{{
  "best_id": "id",
  "reason": "brief reason"
}}
"""

    try:
        selected = call_gpt_json(
            selection_prompt,
            system="You are a careful reranker. You only select one existing candidate. Return valid JSON only.",
            max_tokens=1200,
        )

        best_id = str(selected.get("best_id", "0"))

    except Exception:
        best_id = "0"

    best = next((c for c in candidates if c["id"] == best_id), candidates[0])

    # Critical:
    # Return the original repository comment.
    # No GPT generation happens here.
    return best["text"]

In [ ]:
# Match the baseline stance-evaluation prompt and output schema (baseline cell 35).
def evaluate_stance_with_gpt(output_text, topic, expected_user_stance):
    if not str(output_text).strip():
        predicted, confidence, reason = "empty", 0.0, "empty output"
    else:
        prompt = f"""
Topic:
{topic_name(topic)}

Persona text:
{output_text}

Classify the persona text's stance toward the topic.

Labels:
- "pro": supports the topic/movement/policy/right.
- "anti": opposes the topic/movement/policy/right.
- "unclear": unclear, mixed, irrelevant, or ambiguous.

Return valid JSON only:
{{
  "stance": "pro",
  "confidence": 0.0,
  "reason": "brief explanation"
}}
"""
        try:
            data = call_gpt_json(
                prompt,
                system="You are a careful stance classifier. Return valid JSON only.",
                max_tokens=800,
            )
            predicted = str(data.get("stance", "unclear")).lower().strip()
            if predicted not in ("pro", "anti", "unclear"):
                predicted = "unclear"
            confidence = max(0.0, min(1.0, float(data.get("confidence", 0.0))))
            reason = str(data.get("reason", ""))[:300]
        except Exception as e:
            predicted, confidence, reason = "eval_error", 0.0, repr(e)

    target = opposite_stance(expected_user_stance)
    return {
        "predicted_output_stance": predicted,
        "predicted_persona_stance": predicted,
        "stance_confidence": confidence,
        "stance_eval_reason": reason,
        "stance_reason": reason,
        "target_persona_stance": target,
        "opposite_stance_correct": predicted == target,
        "opposite_stance_accuracy": float(predicted == target) if predicted != "eval_error" else np.nan,
    }


In [ ]:

import json
import hashlib
import os
from pathlib import Path

STANCE_CACHE_PATH = (
    SENSITIVITY_OUT_DIR / "stance_evaluation_cache_v1.json"
)

STANCE_CACHE_VERSION = "baseline_stance_prompt_v1"

if STANCE_CACHE_PATH.exists():
    with open(STANCE_CACHE_PATH, "r", encoding="utf-8") as f:
        STANCE_EVAL_CACHE = json.load(f)
else:
    STANCE_EVAL_CACHE = {}

print("Loaded stance cache entries:", len(STANCE_EVAL_CACHE))

# Keep the original classifier.
_original_evaluate_stance_with_gpt = evaluate_stance_with_gpt

def evaluate_stance_with_gpt(output_text, topic, expected_user_stance):
    topic = normalize_topic(topic)
    expected_user_stance = normalize_stance(expected_user_stance)

    key_payload = {
        "version": STANCE_CACHE_VERSION,
        "model": GPT_MODEL,
        "topic": topic,
        "output_text": str(output_text),
    }

    cache_key = hashlib.sha256(
        json.dumps(
            key_payload, sort_keys=True, ensure_ascii=False
        ).encode("utf-8")
    ).hexdigest()

    if cache_key not in STANCE_EVAL_CACHE:
        result = _original_evaluate_stance_with_gpt(
            output_text, topic, expected_user_stance
        )

        predicted = result["predicted_persona_stance"]

        # Do not permanently cache failed API evaluations.
        if predicted not in ("eval_error",):
            STANCE_EVAL_CACHE[cache_key] = {
                "predicted_persona_stance": predicted,
                "stance_confidence": result.get("stance_confidence", 0.0),
                "stance_reason": result.get("stance_reason", ""),
            }

            tmp_path = STANCE_CACHE_PATH.with_suffix(".tmp")
            with open(tmp_path, "w", encoding="utf-8") as f:
                json.dump(
                    STANCE_EVAL_CACHE, f,
                    ensure_ascii=False, indent=2
                )
            os.replace(tmp_path, STANCE_CACHE_PATH)

    if cache_key in STANCE_EVAL_CACHE:
        cached = STANCE_EVAL_CACHE[cache_key]
        predicted = cached["predicted_persona_stance"]
        target = opposite_stance(expected_user_stance)

        return {
            "predicted_output_stance": predicted,
            "predicted_persona_stance": predicted,
            "stance_confidence": cached["stance_confidence"],
            "stance_eval_reason": cached["stance_reason"],
            "stance_reason": cached["stance_reason"],
            "target_persona_stance": target,
            "opposite_stance_correct": predicted == target,
            "opposite_stance_accuracy": float(predicted == target),
        }

    # Return the original error result if evaluation failed.
    return result

print("Persistent stance evaluation cache is ready.")


In [ ]:
def evaluate_output(user_opinion, output_text, topic, expected_user_stance):
    valid_output = bool(str(output_text).strip())

    return {
        "eval_nli_model": EVAL_NLI_MODEL_NAME,
        "method_nli_model": METHOD_NLI_MODEL_NAME,
        "nli_contradiction": eval_nli_score(user_opinion, output_text) if valid_output else np.nan,
        "valid_output": valid_output,
        "response_words": len(str(output_text).split()) if valid_output else 0,
        **evaluate_stance_with_gpt(output_text, topic, expected_user_stance),
    }

In [ ]:
# ============================================================
# ANALYSIS 3 — SCORE-WEIGHT SENSITIVITY
# Full method only
# All 600 evaluation cases
# ============================================================

from pathlib import Path
import time
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


SENSITIVITY_OUT_DIR = (
    OUT_DIR / "gpt_score_sensitivity_600_final"
)

SENSITIVITY_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SENSITIVITY_RESULTS_PATH = (
    SENSITIVITY_OUT_DIR /
    "gpt_score_sensitivity_full_results_600_final.csv"
)

SENSITIVITY_VARIANT_NAME = "full_method"


# ------------------------------------------------------------
# Weight multipliers
#
# 0.00 = remove this score's direct contribution
# 0.50 = half of original weight
# 1.00 = EXACT ORIGINAL FULL METHOD
# 1.50 = 150% of original weight
# 2.00 = double original weight
# ------------------------------------------------------------

WEIGHT_MULTIPLIERS = [
    0.00,
    0.25,
    0.50,
    0.75,
    1.00,
    1.25,
    1.50,
    2.00,
]


# These are the scores we vary ONE AT A TIME
SENSITIVITY_SCORES = [
    "keep_score",
    "coverage_score",
    "uniqueness_score",
    "persona_usefulness",
    "clarity_score",
    "redundancy_risk",
]


# ------------------------------------------------------------
# ORIGINAL Full Method weights
#
# IMPORTANT:
# These must match build_runtime_pool_score from the
# original evaluation code.
# ------------------------------------------------------------

ORIGINAL_RUNTIME_WEIGHTS = {
    "gpt_final_score": 2.0,
    "keep_score": 1.5,
    "persona_usefulness": 1.2,
    "coverage_score": 1.0,
    "uniqueness_score": 0.8,
    "clarity_score": 0.5,
    "redundancy_risk": -0.8,
    "hnsw_similarity": 0.5,
    "topic_candidate_score": 0.3,
}


print("Analysis:")
print("SCORE-WEIGHT SENSITIVITY")

print("\nOutput:")
print(SENSITIVITY_OUT_DIR)

print("\nScores varied:")
print(SENSITIVITY_SCORES)

print("\nMultipliers:")
print(WEIGHT_MULTIPLIERS)

print(
    "\nIMPORTANT: multiplier=1.0 "
    "must reproduce the original Full Method."
)


In [ ]:
# ============================================================
# USE THE COMPLETE 600-CASE EVALUATION SET
# ============================================================

sensitivity_test_cases_df = (
    test_cases_df
    .copy()
    .reset_index(drop=True)
)

print(
    "Sensitivity test cases:",
    len(sensitivity_test_cases_df)
)

display(
    sensitivity_test_cases_df
    .groupby(
        [
            "topic",
            "expected_user_stance"
        ]
    )
    .size()
    .reset_index(name="count")
)


assert len(sensitivity_test_cases_df) == 600

counts = (
    sensitivity_test_cases_df
    .groupby(
        [
            "topic",
            "expected_user_stance"
        ]
    )
    .size()
)

assert (counts == 100).all()

print(
    "\nSensitivity test-set validation PASSED."
)

In [ ]:
# ============================================================
# VALIDATE FULL-METHOD REPOSITORIES
# ============================================================

for topic in [
    "blm",
    "guns",
    "samesex"
]:

    repo = load_repository(
        topic,
        SENSITIVITY_VARIANT_NAME
    )

    missing = [
        score
        for score in SENSITIVITY_SCORES
        if score not in repo.columns
    ]

    if missing:

        raise ValueError(
            f"{topic}: missing "
            f"{missing}"
        )

    print(
        topic,
        "OK | rows=",
        len(repo)
    )


print(
    "\nRepositories validated."
)

In [ ]:
# ============================================================
# CACHE FULL-METHOD REPOSITORIES
# ============================================================

SENSITIVITY_REPO_CACHE = {}


for topic in [
    "blm",
    "guns",
    "samesex"
]:

    SENSITIVITY_REPO_CACHE[
        topic
    ] = load_repository(
        topic,
        SENSITIVITY_VARIANT_NAME
    )

    print(
        topic,
        "loaded:",
        len(
            SENSITIVITY_REPO_CACHE[
                topic
            ]
        )
    )


print(
    "\nRepository cache ready."
)

In [ ]:
# ============================================================
# PERSISTENT GPT SELECTION CACHE (RESUME-SAFE)
# ============================================================
# IMPORTANT:
# The old notebook kept this cache only in RAM. After a Colab
# restart, the CSV survived but the cache disappeared, so the
# same case + same shortlist could be sent to GPT again and
# receive a different selection. This cache is persisted to
# Drive after every NEW GPT selection.

GPT_SELECTION_CACHE_PATH = (
    SENSITIVITY_OUT_DIR / "gpt_selection_cache_baseline_prompt_v1.json"
)


def _normalize_cache_value(value):
    """Backward/forward compatible cache reader."""
    if isinstance(value, dict):
        return str(value.get("output_text", ""))
    return str(value)


def load_persistent_gpt_cache():
    if not GPT_SELECTION_CACHE_PATH.exists():
        print("No persistent GPT cache found. Starting fresh.")
        return {}

    with open(GPT_SELECTION_CACHE_PATH, "r", encoding="utf-8") as f:
        raw = json.load(f)

    cache = {
        str(k): _normalize_cache_value(v)
        for k, v in raw.items()
    }

    print(
        "Loaded persistent GPT cache:",
        len(cache),
        "entries"
    )
    return cache


def save_persistent_gpt_cache():
    """Atomic-ish save: write temp file, then replace target."""
    tmp_path = GPT_SELECTION_CACHE_PATH.with_suffix(".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(
            GPT_SELECTION_CACHE,
            f,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
        )

    tmp_path.replace(GPT_SELECTION_CACHE_PATH)


GPT_SELECTION_CACHE = load_persistent_gpt_cache()


def make_shortlist_cache_key(
    topic,
    user_opinion,
    expected_user_stance,
    shortlist,
):
    candidate_data = []

    for _, row in shortlist.iterrows():
        candidate_data.append({
            "text": str(row["_text"]),
            "stance_label": str(row.get("stance_label", "unknown")),
            "method_nli_score": round(
                float(row["method_nli_score"]),
                8,
            ),
        })

    payload = {
        "topic": normalize_topic(topic),
        "user_opinion": str(user_opinion).strip(),
        "expected_user_stance": normalize_stance(expected_user_stance),
        "candidates": candidate_data,
    }

    payload_string = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
    )

    return hashlib.sha256(
        payload_string.encode("utf-8")
    ).hexdigest()


print("Persistent GPT shortlist cache ready.")
print("Cache path:", GPT_SELECTION_CACHE_PATH)
print("Current cache size:", len(GPT_SELECTION_CACHE))


In [ ]:
def build_runtime_pool_score_weight_sensitivity(
    candidate_df, varied_score, multiplier
):
    if varied_score not in SENSITIVITY_SCORES:
        raise ValueError(f"Unknown sensitivity score: {varied_score}")

    weights = ORIGINAL_RUNTIME_WEIGHTS.copy()
    weights[varied_score] *= float(multiplier)

    score = pd.Series(
        0.0, index=candidate_df.index, dtype="float64"
    )

    for col, weight in weights.items():
        if col in candidate_df.columns:
            values = pd.to_numeric(
                candidate_df[col], errors="coerce"
            ).fillna(0.0)

            score += float(weight) * values

    return score.to_numpy(dtype=np.float32)

In [ ]:
# ============================================================
# FULL-METHOD ONLINE RETRIEVAL
# WITH ONE MODIFIED RUNTIME WEIGHT
# ============================================================

def generate_persona_response_sensitivity(
    user_opinion,
    topic,
    expected_user_stance,
    score_name,
    multiplier,
):

    topic = normalize_topic(topic)

    expected_user_stance = (
        normalize_stance(
            expected_user_stance
        )
    )

    target_stance = (
        opposite_stance(
            expected_user_stance
        )
    )


    # --------------------------------------------------------
    # ORIGINAL repository — no score modification
    # --------------------------------------------------------

    repo = (
        SENSITIVITY_REPO_CACHE[
            topic
        ]
        .copy()
    )


    pool = repo[
        repo["stance_label"]
        == target_stance
    ].copy()


    if pool.empty:

        return {
            "output_text": "",
            "pool_signature": "",
            "shortlist_signature": "",
        }


    # --------------------------------------------------------
    # Runtime score with ONE modified weight
    # --------------------------------------------------------

    pool[
        "_runtime_pool_score"
    ] = (
        build_runtime_pool_score_weight_sensitivity(

            candidate_df=pool,

            varied_score=
                score_name,

            multiplier=
                multiplier,
        )
    )


    pool = (
        pool
        .sort_values(
            "_runtime_pool_score",
            ascending=False
        )
        .head(POOL_SIZE)
        .copy()
    )


    pool_signature = hashlib.sha256(
        json.dumps(
            pool["_text"]
            .fillna("")
            .astype(str)
            .tolist(),
            ensure_ascii=False
        ).encode("utf-8")
    ).hexdigest()


    candidate_texts = (
        pool["_text"]
        .fillna("")
        .astype(str)
        .tolist()
    )


    if not candidate_texts:

        return {
            "output_text": "",
            "pool_signature":
                pool_signature,
            "shortlist_signature": "",
        }


    # --------------------------------------------------------
    # SAME NLI reranking
    # --------------------------------------------------------

    nli_scores = method_nli_scores(
        user_opinion,
        candidate_texts
    )


    pool[
        "method_nli_score"
    ] = nli_scores


    shortlist = (
        pool
        .sort_values(
            "method_nli_score",
            ascending=False
        )
        .head(K2_SHORT)
        .copy()
    )


    shortlist_signature = (
        hashlib.sha256(
            json.dumps(
                shortlist["_text"]
                .fillna("")
                .astype(str)
                .tolist(),
                ensure_ascii=False
            ).encode("utf-8")
        ).hexdigest()
    )


    if shortlist.empty:

        return {
            "output_text": "",
            "pool_signature":
                pool_signature,
            "shortlist_signature":
                shortlist_signature,
        }


    # --------------------------------------------------------
    # GPT cache
    # --------------------------------------------------------

    cache_key = (
        make_shortlist_cache_key(
            topic=topic,
            user_opinion=user_opinion,
            expected_user_stance=(
                expected_user_stance
            ),
            shortlist=shortlist,
        )
    )


    if cache_key in GPT_SELECTION_CACHE:

        return {
            "output_text":
                GPT_SELECTION_CACHE[
                    cache_key
                ],

            "pool_signature":
                pool_signature,

            "shortlist_signature":
                shortlist_signature,
        }


    # --------------------------------------------------------
    # Candidates for GPT
    # --------------------------------------------------------

    # EXACT baseline GPT candidate format and prompt (baseline cell 30).
    candidates = []
    for i, row in shortlist.iterrows():
        candidates.append({
            "id": str(i),
            "text": str(row["_text"])[:700],
            "nli_score": float(row["method_nli_score"]),
        })

    selection_prompt = f"""
Topic: {topic_name(topic)}

User opinion:
{user_opinion}

Candidate opposite-stance comments:
{json.dumps(candidates, ensure_ascii=False)}

Select the single best comment for constructing an opposite-opinion persona.
Prefer direct contradiction, topic relevance, clarity, and persona usefulness.

Return a valid JSON object with this exact schema:
{{
  "best_id": "id",
  "reason": "brief reason"
}}
Do not output any text outside the JSON object.
"""

    try:
        selected = call_gpt_json(
            selection_prompt,
            system="You are a careful contradiction reranker. Return valid JSON only.",
            max_tokens=1200,
        )
        best_id = str(selected.get("best_id", candidates[0]["id"]))
    except Exception as e:
        print("GPT selection failed, using top NLI candidate:", repr(e))
        best_id = candidates[0]["id"]

    best = next(
        (
            candidate
            for candidate in candidates
            if candidate["id"]
            == best_id
        ),
        candidates[0],
    )


    output_text = (
        best["text"]
    )


    GPT_SELECTION_CACHE[
        cache_key
    ] = output_text

    # CRITICAL: persist immediately so a Colab restart cannot
    # cause the same shortlist to be reranked by GPT again.
    save_persistent_gpt_cache()


    return {

        "output_text":
            output_text,

        "pool_signature":
            pool_signature,

        "shortlist_signature":
            shortlist_signature,
    }


In [ ]:
# ============================================================
# RUN ONE WEIGHT-SENSITIVITY CASE
# ============================================================

def run_single_sensitivity_case(
    case_row,
    score_name,
    multiplier,
):

    topic = normalize_topic(
        case_row["topic"]
    )

    user_opinion = str(
        case_row["user_opinion"]
    )

    expected_user_stance = (
        normalize_stance(
            case_row[
                "expected_user_stance"
            ]
        )
    )


    start = time.time()

    method_error = ""


    try:

        generated = (
            generate_persona_response_sensitivity(

                user_opinion=
                    user_opinion,

                topic=
                    topic,

                expected_user_stance=
                    expected_user_stance,

                score_name=
                    score_name,

                multiplier=
                    multiplier,
            )
        )


        output_text = (
            generated[
                "output_text"
            ]
        )


    except Exception as e:

        output_text = ""

        method_error = repr(e)

        generated = {
            "pool_signature": "",
            "shortlist_signature": "",
        }


    runtime_seconds = (
        time.time() - start
    )


    try:

        metrics = evaluate_output(
            user_opinion=user_opinion,
            output_text=output_text,
            topic=topic,
            expected_user_stance=(
                expected_user_stance
            ),
        )


    except Exception as e:

        metrics = {

            "eval_nli_model":
                EVAL_NLI_MODEL_NAME,

            "method_nli_model":
                METHOD_NLI_MODEL_NAME,

            "nli_contradiction":
                np.nan,

            "valid_output":
                False,

            "response_words":
                0,

            "predicted_output_stance":
                "eval_error",

            "opposite_stance_accuracy":
                np.nan,

            "stance_eval_reason":
                repr(e),
        }


    original_weight = (
        ORIGINAL_RUNTIME_WEIGHTS[
            score_name
        ]
    )


    effective_weight = (
        original_weight
        * float(multiplier)
    )


    return {

        "topic":
            topic,

        "user_opinion":
            user_opinion,

        "expected_user_stance":
            expected_user_stance,

        "target_persona_stance":
            opposite_stance(
                expected_user_stance
            ),

        "variant":
            SENSITIVITY_VARIANT_NAME,

        "score_name":
            score_name,

        "multiplier":
            float(multiplier),

        "original_weight":
            float(original_weight),

        "effective_weight":
            float(effective_weight),

        "runtime_seconds":
            runtime_seconds,

        "runtime_ms":
            runtime_seconds * 1000,

        "output_text":
            output_text,

        "method_error":
            method_error,

        "pool_signature":
            generated[
                "pool_signature"
            ],

        "shortlist_signature":
            generated[
                "shortlist_signature"
            ],

        **metrics,
    }

In [ ]:
# ============================================================
# EXPECTED RUN SIZE
# ============================================================

n_scores = len(
    SENSITIVITY_SCORES
)

n_multipliers = len(
    WEIGHT_MULTIPLIERS
)

n_cases = len(
    sensitivity_test_cases_df
)


expected_runs = (
    n_scores
    * n_multipliers
    * n_cases
)


print(
    "Scores:",
    n_scores
)

print(
    "Multipliers:",
    n_multipliers
)

print(
    "Cases:",
    n_cases
)

print(
    "Total evaluations:",
    expected_runs
)

In [ ]:
# NOTE FOR SENSITIVITY RUN:
# This notebook writes to a NEW v4 results folder/file and a NEW
# persistent GPT cache. Do not copy the previous results
# into this folder. Safe resume is supported automatically.
print("SENSITIVITY results path:", SENSITIVITY_RESULTS_PATH)
print("Persistent GPT cache path:", GPT_SELECTION_CACHE_PATH)
print("Persistent GPT cache entries:", len(GPT_SELECTION_CACHE))

# ============================================================
# RUN FULL SCORE-WEIGHT SENSITIVITY WITH SAFE RESUME
# ============================================================

# ------------------------------------------------------------
# 1. Load existing results
# ------------------------------------------------------------

if SENSITIVITY_RESULTS_PATH.exists():

    existing_df = pd.read_csv(
        SENSITIVITY_RESULTS_PATH,
        encoding="utf-8-sig",
    )

    print(
        "Loaded existing results:",
        existing_df.shape
    )

else:

    existing_df = pd.DataFrame()

    print(
        "No existing results found."
    )


# ------------------------------------------------------------
# 2. Remove accidental duplicates
# ------------------------------------------------------------

RESUME_KEY_COLS = [
    "topic",
    "expected_user_stance",
    "user_opinion",
    "score_name",
    "multiplier",
]


if not existing_df.empty:

    before_dedup = len(existing_df)

    existing_df = (
        existing_df
        .drop_duplicates(
            subset=RESUME_KEY_COLS,
            keep="first",
        )
        .reset_index(drop=True)
    )

    after_dedup = len(existing_df)

    print(
        "Rows before duplicate cleanup:",
        before_dedup
    )

    print(
        "Rows after duplicate cleanup:",
        after_dedup
    )

    print(
        "Duplicates removed:",
        before_dedup - after_dedup
    )

    # Save cleaned file immediately
    existing_df.to_csv(
        SENSITIVITY_RESULTS_PATH,
        index=False,
        encoding="utf-8-sig",
    )


# ------------------------------------------------------------
# 3. Convert saved rows to records
# ------------------------------------------------------------

existing_rows = (
    existing_df.to_dict("records")
    if not existing_df.empty
    else []
)


# ------------------------------------------------------------
# 4. Build SAFE resume keys
#
# multiplier is ALWAYS float.
# topic/stance are normalized.
# user opinion is stripped.
# ------------------------------------------------------------

done_keys = {

    (
        normalize_topic(
            row["topic"]
        ),

        normalize_stance(
            row[
                "expected_user_stance"
            ]
        ),

        str(
            row["user_opinion"]
        ).strip(),

        str(
            row["score_name"]
        ).strip(),

        float(
            row["multiplier"]
        ),
    )

    for row in existing_rows
}


print(
    "\nExisting saved rows:",
    len(existing_rows)
)

print(
    "Resume keys:",
    len(done_keys)
)


if existing_rows and len(done_keys) == 0:

    raise RuntimeError(
        "Resume failed: results were loaded "
        "but no resume keys were created."
    )


# ------------------------------------------------------------
# 5. Show exactly what is already complete
# ------------------------------------------------------------

if not existing_df.empty:

    progress = (
        existing_df
        .groupby(
            [
                "score_name",
                "multiplier"
            ]
        )
        .size()
        .reset_index(
            name="completed_cases"
        )
    )

    print(
        "\nCurrent saved progress:"
    )

    display(progress)


# ------------------------------------------------------------
# 6. Continue only missing cases
# ------------------------------------------------------------

rows = existing_rows.copy()


for score_name in SENSITIVITY_SCORES:

    print(
        "\n======================================"
    )

    print(
        "SCORE:",
        score_name
    )

    print(
        "======================================"
    )


    for multiplier in WEIGHT_MULTIPLIERS:

        multiplier = float(
            multiplier
        )


        # ----------------------------------------------------
        # Count existing cases for this exact configuration
        # ----------------------------------------------------

        already_done = sum(

            1

            for key in done_keys

            if (
                key[3] == score_name
                and
                key[4] == multiplier
            )
        )


        print(
            f"\n--- {score_name} | "
            f"multiplier={multiplier} ---"
        )

        print(
            f"Already completed: "
            f"{already_done}/"
            f"{len(sensitivity_test_cases_df)}"
        )


        # Whole multiplier already finished
        if already_done >= len(
            sensitivity_test_cases_df
        ):

            print(
                "Already complete -> skipping."
            )

            continue


        # ----------------------------------------------------
        # Iterate through 600 cases,
        # but execute ONLY missing ones
        # ----------------------------------------------------

        for _, case_row in tqdm(

            sensitivity_test_cases_df
            .iterrows(),

            total=len(
                sensitivity_test_cases_df
            ),

            desc=(
                f"{score_name} "
                f"x{multiplier}"
            ),
        ):


            key = (

                normalize_topic(
                    case_row["topic"]
                ),

                normalize_stance(
                    case_row[
                        "expected_user_stance"
                    ]
                ),

                str(
                    case_row[
                        "user_opinion"
                    ]
                ).strip(),

                str(
                    score_name
                ).strip(),

                float(
                    multiplier
                ),
            )


            # Already saved -> skip
            if key in done_keys:
                continue


            result = (
                run_single_sensitivity_case(

                    case_row=
                        case_row,

                    score_name=
                        score_name,

                    multiplier=
                        multiplier,
                )
            )


            rows.append(
                result
            )

            done_keys.add(
                key
            )


            # Save after EVERY newly completed case
            pd.DataFrame(
                rows
            ).to_csv(

                SENSITIVITY_RESULTS_PATH,

                index=False,

                encoding="utf-8-sig",
            )


# ------------------------------------------------------------
# 7. Finished / current state
# ------------------------------------------------------------

sensitivity_results_df = (
    pd.DataFrame(rows)
)


print(
    "\n======================================"
)

print(
    "RUN FINISHED"
)

print(
    "======================================"
)

print(
    "Saved:",
    SENSITIVITY_RESULTS_PATH
)

print(
    "Rows:",
    len(
        sensitivity_results_df
    )
)

print(
    "Unique resume keys:",
    len(done_keys)
)


In [ ]:
# ============================================================
# VALIDATE COMPLETENESS
# ============================================================

expected_n = (
    len(SENSITIVITY_SCORES)
    *
    len(WEIGHT_MULTIPLIERS)
    *
    len(sensitivity_test_cases_df)
)


actual_n = len(
    sensitivity_results_df
)


duplicates = (

    sensitivity_results_df
    .duplicated(
        subset=[
            "topic",
            "expected_user_stance",
            "user_opinion",
            "score_name",
            "multiplier",
        ]
    )
    .sum()
)


print(
    "Expected rows:",
    expected_n
)

print(
    "Actual rows:",
    actual_n
)

print(
    "Duplicates:",
    duplicates
)


if (
    actual_n == expected_n
    and duplicates == 0
):

    print(
        "\nWeight sensitivity "
        "run COMPLETE."
    )

else:

    print(
        "\nRun is NOT complete."
    )

In [ ]:
# ============================================================
# OVERALL WEIGHT-SENSITIVITY SUMMARY
# + RETRIEVAL STABILITY
# ============================================================

sensitivity_results_df = (
    pd.read_csv(
        SENSITIVITY_RESULTS_PATH,
        encoding="utf-8-sig",
    )
)


# ------------------------------------------------------------
# ORIGINAL FULL METHOD = multiplier 1.0
# ------------------------------------------------------------

baseline = (

    sensitivity_results_df[
        sensitivity_results_df[
            "multiplier"
        ] == 1.0
    ][
        [
            "topic",
            "user_opinion",
            "expected_user_stance",
            "score_name",
            "pool_signature",
            "shortlist_signature",
        ]
    ]

    .rename(
        columns={
            "pool_signature":
                "original_pool_signature",

            "shortlist_signature":
                "original_shortlist_signature",
        }
    )
)


sensitivity_results_df = (
    sensitivity_results_df
    .merge(
        baseline,

        on=[
            "topic",
            "user_opinion",
            "expected_user_stance",
            "score_name",
        ],

        how="left",
    )
)


sensitivity_results_df[
    "pool_changed_vs_original"
] = (
    sensitivity_results_df[
        "pool_signature"
    ]
    !=
    sensitivity_results_df[
        "original_pool_signature"
    ]
)


sensitivity_results_df[
    "shortlist_changed_vs_original"
] = (
    sensitivity_results_df[
        "shortlist_signature"
    ]
    !=
    sensitivity_results_df[
        "original_shortlist_signature"
    ]
)


# Sanity check
original_rows = (
    sensitivity_results_df[
        sensitivity_results_df[
            "multiplier"
        ] == 1.0
    ]
)


assert (
    original_rows[
        "pool_changed_vs_original"
    ].sum()
    == 0
)

assert (
    original_rows[
        "shortlist_changed_vs_original"
    ].sum()
    == 0
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

sensitivity_summary = (

    sensitivity_results_df
    .groupby(
        [
            "score_name",
            "multiplier"
        ],
        as_index=False,
    )
    .agg(

        n_cases=(
            "user_opinion",
            "size"
        ),

        original_weight=(
            "original_weight",
            "first"
        ),

        effective_weight=(
            "effective_weight",
            "first"
        ),

        mean_nli_contradiction=(
            "nli_contradiction",
            "mean"
        ),

        opposite_stance_accuracy=(
            "opposite_stance_accuracy",
            "mean"
        ),

        pool_change_rate=(
            "pool_changed_vs_original",
            "mean"
        ),

        shortlist_change_rate=(
            "shortlist_changed_vs_original",
            "mean"
        ),

        mean_runtime_seconds=(
            "runtime_seconds",
            "mean"
        ),

        errors=(
            "method_error",

            lambda x: int(
                (
                    x
                    .fillna("")
                    .astype(str)
                    .str.strip()
                    != ""
                ).sum()
            )
        ),
    )
)


sensitivity_summary[
    "pool_change_percent"
] = (
    sensitivity_summary[
        "pool_change_rate"
    ]
    * 100
)


sensitivity_summary[
    "shortlist_change_percent"
] = (
    sensitivity_summary[
        "shortlist_change_rate"
    ]
    * 100
)


SUMMARY_PATH = (
    SENSITIVITY_OUT_DIR /
    "score_weight_summary_600.csv"
)


sensitivity_summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)


print(
    "Saved:",
    SUMMARY_PATH
)


display(
    sensitivity_summary[
        [
            "score_name",
            "multiplier",
            "original_weight",
            "effective_weight",
            "mean_nli_contradiction",
            "opposite_stance_accuracy",
            "pool_change_percent",
            "shortlist_change_percent",
            "errors",
        ]
    ]
)

In [ ]:
# ============================================================
# WEIGHT SENSITIVITY BY TOPIC
# ============================================================

sensitivity_by_topic = (

    sensitivity_results_df
    .groupby(
        [
            "topic",
            "score_name",
            "multiplier"
        ],
        as_index=False,
    )
    .agg(

        n_cases=(
            "user_opinion",
            "size"
        ),

        mean_nli_contradiction=(
            "nli_contradiction",
            "mean"
        ),

        opposite_stance_accuracy=(
            "opposite_stance_accuracy",
            "mean"
        ),

        pool_change_rate=(
            "pool_changed_vs_original",
            "mean"
        ),

        shortlist_change_rate=(
            "shortlist_changed_vs_original",
            "mean"
        ),
    )
)


BY_TOPIC_PATH = (
    SENSITIVITY_OUT_DIR /
    "score_weight_by_topic_600.csv"
)


sensitivity_by_topic.to_csv(
    BY_TOPIC_PATH,
    index=False,
    encoding="utf-8-sig",
)


print(
    "Saved:",
    BY_TOPIC_PATH
)


display(
    sensitivity_by_topic
)

In [ ]:
# ============================================================
# WEIGHT SENSITIVITY BY TOPIC + USER STANCE
# ============================================================

sensitivity_by_topic_stance = (

    sensitivity_results_df
    .groupby(
        [
            "topic",
            "expected_user_stance",
            "score_name",
            "multiplier"
        ],
        as_index=False,
    )
    .agg(

        n_cases=(
            "user_opinion",
            "size"
        ),

        mean_nli_contradiction=(
            "nli_contradiction",
            "mean"
        ),

        opposite_stance_accuracy=(
            "opposite_stance_accuracy",
            "mean"
        ),

        pool_change_rate=(
            "pool_changed_vs_original",
            "mean"
        ),

        shortlist_change_rate=(
            "shortlist_changed_vs_original",
            "mean"
        ),
    )
)


BY_TOPIC_STANCE_PATH = (
    SENSITIVITY_OUT_DIR /
    "score_weight_by_topic_and_stance_600.csv"
)


sensitivity_by_topic_stance.to_csv(
    BY_TOPIC_STANCE_PATH,
    index=False,
    encoding="utf-8-sig",
)


print(
    "Saved:",
    BY_TOPIC_STANCE_PATH
)


display(
    sensitivity_by_topic_stance
)

In [ ]:
# ============================================================
# SCORE-WEIGHT SENSITIVITY PLOTS
# ============================================================

PLOT_DIR = (
    SENSITIVITY_OUT_DIR /
    "plots"
)

PLOT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def plot_weight_metric(
    summary_df,
    metric_col,
    y_label,
):

    for score_name in (
        SENSITIVITY_SCORES
    ):


        sub = (

            summary_df[
                summary_df[
                    "score_name"
                ]
                == score_name
            ]

            .sort_values(
                "multiplier"
            )

            .copy()
        )


        plt.figure(
            figsize=(8, 5)
        )


        plt.plot(
            sub["multiplier"],
            sub[metric_col],
            marker="o",
            linewidth=2,
        )


        # Original Full Method
        plt.axvline(
            x=1.0,
            linestyle="--",
            alpha=0.7,
            label=(
                "Original Full Method "
                "(x1.0)"
            ),
        )


        plt.xlabel(
            "Original weight multiplier"
        )

        plt.ylabel(
            y_label
        )

        plt.title(
            f"{score_name}: "
            f"{y_label}"
        )

        plt.xticks(
            sub["multiplier"]
        )

        plt.grid(
            True,
            alpha=0.3
        )

        plt.legend()

        plt.tight_layout()


        out_path = (
            PLOT_DIR /
            f"{metric_col}_"
            f"weight_"
            f"{score_name}.png"
        )


        plt.savefig(
            out_path,
            dpi=300,
            bbox_inches="tight",
        )

        plt.show()


# Performance
plot_weight_metric(
    sensitivity_summary,
    "mean_nli_contradiction",
    "Mean NLI contradiction",
)


plot_weight_metric(
    sensitivity_summary,
    "opposite_stance_accuracy",
    "Opposite stance accuracy",
)


# Retrieval stability
plot_weight_metric(
    sensitivity_summary,
    "pool_change_percent",
    "Top-60 change vs original (%)",
)


plot_weight_metric(
    sensitivity_summary,
    "shortlist_change_percent",
    "Top-8 change vs original (%)",
)